In [1]:
import json
import re
import html
import unicodedata
from pathlib import Path

import pandas as pd

SEED = 42
BASE_PATH = Path("data/MeEval")
OUT_DIR = Path("processed/mr_en")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def load_files(folder, prefix, source):
    frames = []
    for split in ["train", "val", "test"]:
        path = BASE_PATH / folder / f"{prefix}-{split}.csv"
        data = pd.read_csv(path)[["tweet", "hate", "sentiment", "profanity"]].copy()
        data["source"] = source
        data["split"] = split
        frames.append(data)
    return pd.concat(frames, ignore_index=True)


mesent = load_files("L3Cube-MeSent", "codemix-sent", "MeSent")
mehate = load_files("L3Cube-MeHate", "codemix-hate-small", "MeHate")
df = pd.concat([mesent, mehate], ignore_index=True)

print("MeSent:", len(mesent))
print("MeHate:", len(mehate))

MeSent: 12000
MeHate: 2768


In [3]:
def clean_text(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200b", "")
    text = re.sub(r"https?://\S+|www\.\S+", "URL", text)
    text = re.sub(r"(@USER\s*){2,}", "@USER ", text)
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def make_key(text):
    # only used to find duplicates, never given to the model
    text = text.lower().replace("@user", " ")
    text = re.sub(r"\burl\b", " ", text)
    return "".join(c for c in text if c.isalnum() or unicodedata.category(c).startswith("M"))


for col in ["hate", "sentiment", "profanity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["tweet", "hate", "sentiment", "profanity"])
df = df[df["hate"].isin([0, 1]) & df["profanity"].isin([0, 1]) & df["sentiment"].isin([-1, 0, 1])]
df = df.astype({"hate": int, "sentiment": int, "profanity": int})

df["tweet"] = df["tweet"].map(clean_text)
df["key"] = df["tweet"].map(make_key)
df = df[df["key"] != ""]

print("Rows after cleaning:", len(df))
print("Unique tweets:", df["key"].nunique())

Rows after cleaning: 14764
Unique tweets: 11982


In [4]:
df["labels"] = df["sentiment"].astype(str) + "_" + df["hate"].astype(str) + "_" + df["profanity"].astype(str)
labels_per_tweet = df.groupby("key")["labels"].nunique()
conflict_keys = labels_per_tweet[labels_per_tweet > 1].index

df[df["key"].isin(conflict_keys)].to_csv(OUT_DIR / "dropped_conflicts.csv", index=False)
df = df[~df["key"].isin(conflict_keys)]
print("Tweets with conflicting labels dropped:", len(conflict_keys))


def join_sources(sources):
    return "+".join(sorted(set(sources)))


df["source"] = df.groupby("key")["source"].transform(join_sources)

priority = {"test": 3, "val": 2, "train": 1}
df["priority"] = df["split"].map(priority)
df = df.sort_values("priority", ascending=False, kind="stable")
df = df.drop_duplicates("key")

print("Unique tweets kept:", len(df))

Tweets with conflicting labels dropped: 14
Unique tweets kept: 11968


In [5]:
df["language"] = "mr-en"
df["sentiment_label"] = df["sentiment"].map({-1: "negative", 0: "neutral", 1: "positive"})
df["offensive"] = ((df["hate"] == 1) | (df["profanity"] == 1)).astype(int)
df["n_words"] = df["tweet"].str.split().str.len()


def create_output(row):
    return json.dumps({
        "language": row["language"],
        "sentiment": row["sentiment_label"],
        "hate": bool(row["hate"]),
        "profanity": bool(row["profanity"]),
    })


df["output"] = df.apply(create_output, axis=1)


def take(split):
    part = df[df["split"] == split]
    return part.sample(frac=1, random_state=SEED).reset_index(drop=True)


train_df = take("train")
val_df = take("val")
test_df = take("test")
splits = {"train": train_df, "val": val_df, "test": test_df}

In [6]:
train_keys = set(train_df["key"])
val_keys = set(val_df["key"])
test_keys = set(test_df["key"])

assert not (train_keys & val_keys), "train and val share tweets"
assert not (train_keys & test_keys), "train and test share tweets"
assert not (val_keys & test_keys), "val and test share tweets"

for name, data in splits.items():
    negative = (data["sentiment"] == -1).mean() * 100
    neutral = (data["sentiment"] == 0).mean() * 100
    positive = (data["sentiment"] == 1).mean() * 100
    hate = data["hate"].mean() * 100
    profanity = data["profanity"].mean() * 100
    offensive = data["offensive"].mean() * 100
    print(f"{name:<6} rows={len(data):<6} neg={negative:.1f}% neu={neutral:.1f}% pos={positive:.1f}% "
          f"hate={hate:.1f}% profanity={profanity:.1f}% offensive={offensive:.1f}%")

train  rows=8811   neg=33.0% neu=33.0% pos=34.0% hate=8.8% profanity=2.1% offensive=9.3%
val    rows=1535   neg=39.4% neu=30.1% pos=30.5% hate=18.6% profanity=5.2% offensive=19.3%
test   rows=1622   neg=40.9% neu=28.9% pos=30.3% hate=19.5% profanity=4.7% offensive=20.2%


In [7]:
SYSTEM_PROMPT = (
    "You analyze code-mixed Marathi-English social media text (Roman or Devanagari script). "
    "Reply with only a JSON object in exactly this format: "
    '{"language": "mr-en", "sentiment": "positive|negative|neutral", "hate": true|false, "profanity": true|false}. '
    'Set "hate" to true for hate speech and "profanity" to true for profane language, otherwise false.'
)
USER_TEMPLATE = "Analyze this text.\nText: {text}"

columns = ["tweet", "output", "language", "sentiment_label", "hate", "profanity", "offensive", "source", "n_words"]

for name, data in splits.items():
    rows = []
    for tweet, output in zip(data["tweet"], data["output"]):
        rows.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_TEMPLATE.format(text=tweet)},
            ],
            "completion": [{"role": "assistant", "content": output}],
        })

    with open(OUT_DIR / f"{name}.jsonl", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    data[columns].to_csv(OUT_DIR / f"{name}_full.csv", index=False, encoding="utf-8")

with open(OUT_DIR / "prompt_config.json", "w", encoding="utf-8") as f:
    json.dump({"system_prompt": SYSTEM_PROMPT, "user_template": USER_TEMPLATE}, f, ensure_ascii=False, indent=2)

print("Saved to:", OUT_DIR.resolve())
print(sorted(p.name for p in OUT_DIR.iterdir()))

Saved to: C:\Users\ASUS TUF\OneDrive\Desktop\NLP_PROJECT\processed\mr_en
['dropped_conflicts.csv', 'prompt_config.json', 'test.jsonl', 'test_full.csv', 'test_messages.jsonl', 'train.jsonl', 'train_full.csv', 'train_messages.jsonl', 'val.jsonl', 'val_full.csv', 'val_messages.jsonl']


In [8]:
for name, data in splits.items():
    saved_sentiments = [json.loads(o)["sentiment"] for o in data["output"]]
    saved_hate = [json.loads(o)["hate"] for o in data["output"]]
    assert saved_sentiments == list(data["sentiment_label"])
    assert saved_hate == [bool(h) for h in data["hate"]]

print("Saved JSON matches the labels in every split")
print(json.dumps(train_df.loc[0, ["tweet", "output"]].to_dict(), ensure_ascii=False, indent=2))

Saved JSON matches the labels in every split
{
  "tweet": "@USER Aho mi tumhala magech bolalo hoto mi jar gelo tar mi pahile mazech baghun gheil ;) tyala prob vhyayacha mag :D @USER",
  "output": "{\"language\": \"mr-en\", \"sentiment\": \"negative\", \"hate\": false, \"profanity\": false}"
}
